In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from datasets import load_dataset
dataset_name = "ridwanFatur98/synth-invoice"
train_dataset = load_dataset(dataset_name, split="train")
sample = train_dataset[0]

In [3]:
from scripts.image_process import process_image, image_to_tensor
from scripts.ground_truth_process import process_ground_truth, get_train_inputs

In [4]:
from scripts.tokenizer import get_tokenizer
tokenizer = get_tokenizer()

In [5]:
from scripts.model import DonutModel, DonutConfig

W0930 18:47:43.095000 18425 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0930 18:47:43.128000 18425 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [6]:
# Encoder Config
encoder_layer = [1, 1, 1, 1]
window_size = 10
input_size = [1280, 960]

# Decoder Config
max_position_embeddings = 768
decoder_layer = 1

In [7]:
config = DonutConfig(
    input_size=(1280, 960),
    window_size=10,
    encoder_layer=(1, 1, 1, 1),
    max_position_embeddings=768,
    decoder_layer=1,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
)

In [8]:
donut_model = DonutModel(config).to(device)

In [9]:
sum(p.numel() for p in donut_model.parameters())

96063388

In [10]:
state_dict = torch.load(
    "donut_model.pth",
    map_location=device,
)

donut_model.load_state_dict(state_dict)

<All keys matched successfully>

### Simple Train

In [11]:
lr = 3e-5

In [12]:
optimizer = torch.optim.Adam(donut_model.parameters(), lr=lr)

In [13]:
# Input Config
input_size = [1280, 960]
align_long_axis = False
random_padding = True

# Input Encoder
img = process_image(sample["image"], input_size, align_long_axis, random_padding)
image_tensors = image_to_tensor(img)
image_tensors = image_tensors.unsqueeze(0)
image_tensors = image_tensors.to(device)

# Input Decoder
task_start_token = '<s_cord-v2>'
prompt_end_token= '<s_cord-v2>'

max_length = 768
ignore_id: int = -100
prompt_end_token_id = tokenizer.convert_tokens_to_ids(prompt_end_token)

ground_truth = sample["ground_truth"]
gt_token_sequence, _ = process_ground_truth(
    tokenizer,
    ground_truth,
    task_start_token=task_start_token,
)
processed_parse = gt_token_sequence 

input_ids, labels = get_train_inputs(tokenizer, processed_parse, prompt_end_token_id, max_length, ignore_id)
input_ids = input_ids.unsqueeze(0).to(device)
labels = labels.unsqueeze(0).to(device)

decoder_input_ids = input_ids[:, :-1]
decoder_labels = labels[:, 1:]

In [14]:
num_epochs = 1

for epoch in range(num_epochs):
    optimizer.zero_grad()   
    decoder_outputs = donut_model(
        image_tensors=image_tensors,
        decoder_input_ids=decoder_input_ids,
        decoder_labels=decoder_labels,
    )    
    loss = decoder_outputs['loss']
    print(f"Train - {epoch}: {loss.item()}")
    loss.backward()
    
    optimizer.step()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Train - 0: 0.03186003863811493


In [20]:
torch.save(donut_model.state_dict(), "donut_model.pth")